In [1]:
import pandas as pd
from sqlalchemy import text, create_engine
from config import DB_URL

In [ ]:
# Extract apartment sales data from csv and load to db
create_apartment_sales_table_sql = '''
CREATE TABLE IF NOT EXISTS seoul_apartment_sales (
    month INT,
    district VARCHAR(50),
    units_sold INT,
    PRIMARY KEY (month, district)
)
'''

insert_apartment_sales_sql = '''
INSERT INTO seoul_apartment_sales (
    month,
    district,
    units_sold
) VALUES (
    :month,
    :district,
    :units_sold
)
ON CONFLICT (month, district) DO NOTHING
'''

def get_apartment_sales_records():
    df = pd.read_csv('/workspaces/korea-real-estate-population-movement/data/seoul_apartments_sold.csv')
    df = df[2:].reset_index(drop=True).drop(columns=[
        "자치구별(1)"
        ]).rename(columns={
        "자치구별(2)":"district"
        })

    df = df.melt(
        id_vars=["district"],
        var_name="month",
        value_name="units_sold")

    df["month"] = pd.to_datetime(
        df["month"].str.strip(),
        format="%Y. %m"
    ).dt.strftime("%Y%m").astype(int)

    return df.to_dict(orient="records")

def apartment_sales_main():
    engine = create_engine(DB_URL)
    records = get_apartment_sales_records()

    with engine.begin() as conn:
        conn.execute(text(create_apartment_sales_table_sql))
        conn.execute(text(insert_apartment_sales_sql),records)

In [ ]:
# Extract district code dim table from csv and load to db
create_district_dim_table_sql = '''
CREATE TABLE IF NOT EXISTS district_code_dim (
    code varchar(10),
    district varchar(50),
    PRIMARY KEY(code, district)
) 
'''

insert_district_code_sql = '''
INSERT INTO district_code_dim (
    code,
    district
) VALUES (
    :code,
    :district
) ON CONFLICT DO NOTHING
'''

def get_district_codes_records():
    df = pd.read_csv('/workspaces/korea-real-estate-population-movement/data/seoul_district_codes.csv')
    df['Code'] = df['Code'].astype(str)
    df = df.rename(columns={
        "District":"district",
        "Code":"code"})

    return df.to_dict(orient='records')

def create_district_dim_table(conn):
    conn.execute(text(create_district_dim_table_sql))

def insert_district_codes(conn):
    records = get_district_codes_records()

    conn.execute(text(insert_district_code_sql),records)

def district_code_main():
    engine = create_engine(DB_URL)

    with engine.begin() as conn:
        create_district_dim_table(conn)
        insert_district_codes(conn)

In [2]:
from config import SERVICE_KEY, APT_SALES_BASE_URL, APT_SALES_COLUMNS, END_DATE
import requests
import xml.etree.ElementTree as ET
import pandas as pd

param = {
    'LAWD_CD':11680,
    'DEAL_YMD':202301,
    'serviceKey': SERVICE_KEY,
    'numOfRows':500,
    'pageNo':1
}


response = requests.get(APT_SALES_BASE_URL,params=param,timeout=30)
response.raise_for_status()

#Why do we turn it into response.text and use fromstring
root = ET.fromstring(response.text)


In [7]:
import time
from requests.exceptions import ReadTimeout

create_individual_apt_sales_table_sql = '''
CREATE TABLE IF NOT EXISTS individual_apt_sales (
    id SERIAL PRIMARY KEY,
    apt_name varchar(50),
    build_year varchar(4),
    deal_date varchar(6),
    floor varchar(3),
    district_code varchar(5)
)
'''
insert_individual_apt_sales_sql = '''
INSERT INTO individual_apt_sales (
    apt_name,
    build_year,
    deal_date,
    floor,
    district_code
) VALUES (
    :apt_name,
    :build_year,
    :deal_date,
    :floor,
    :district_code
) ON CONFLICT DO NOTHING
'''

def create_individual_apt_sales_table(conn):
    conn.execute(text(
        create_individual_apt_sales_table_sql))

def insert_individual_apt_sales(conn,records):
    conn.execute(text(
        insert_individual_apt_sales_sql),
        records)


def read_district_code_five_csv():
    df = pd.read_csv('/workspaces/korea-real-estate-population-movement/data/seoul_district_codes.csv')

    required = {"District","Code"}

    if not required.issubset(df.columns):
        raise ValueError("CSV missing required columns.")

    df["District"] = df["District"].str.strip()
    df["Code"] = df["Code"].astype(str).str[:5]
    return dict(zip(df['District'],df['Code']))

def return_df(root):
    items = root.findall(".//item")
    data = []

    for item in items:
        row = {}

        for child in item:
            if child.tag in APT_SALES_COLUMNS:
                row[child.tag] = child.text.strip() if child.text else None

        data.append(row)

    return pd.DataFrame(data)

def rename_df(df):
    return df.rename(columns={
        "aptNm":"apt_name",
        "sggCd":"district_code",
        "buildYear":"build_year"
    })

def convert_date_column(df):
    df["deal_date"] = (
        df["dealYear"].astype(str)
        + df["dealMonth"].astype(str).str.zfill(2)
    )

    return df.drop(columns=["dealYear","dealMonth"])

def individual_sales_apartment_main():
    max_retries = 3
    district_info = read_district_code_five_csv()

    engine = create_engine(DB_URL,
                           pool_pre_ping=True)
    with engine.begin() as conn:
        create_individual_apt_sales_table(conn)

    current_date = pd.to_datetime('202303',format='%Y%m')
    end_date = pd.to_datetime(END_DATE,format='%Y%m')
    while current_date <= end_date:
        data = []
        date = current_date.strftime(format='%Y%m')

        for district in district_info:

            param = {
            'LAWD_CD':district_info[district],
            'DEAL_YMD':date,
            'serviceKey': SERVICE_KEY,
            'numOfRows':300,
            'pageNo':1 
            }

            for attempt in range(max_retries):
                try:
                    response = requests.get(
                        APT_SALES_BASE_URL,
                        params=param,
                        timeout=30)
                    response.raise_for_status()

                    root = ET.fromstring(response.text)

                    total_rows = int(root.findtext('.//totalCount', default='0'))

                    if total_rows > 300:
                        print(f'attempt {attempt+1}: Total_rows exceeds 200: attempting again...'
                              f'numOfRows has been set to {total_rows}')

                        param = {
                        'LAWD_CD':district_info[district],
                        'DEAL_YMD':date,
                        'serviceKey': SERVICE_KEY,
                        'numOfRows':total_rows,
                        'pageNo':1 
                        }

                        response = requests.get(
                            APT_SALES_BASE_URL,
                            params=param,
                            timeout=30)
                        response.raise_for_status()

                        root = ET.fromstring(response.text)

                    df = rename_df(
                        convert_date_column(
                            return_df(root)
                        )
                    )

                    print(f"fetched {district} info")

                    data.append(df)
                    break

                except ReadTimeout:
                    print(
                        f"Timeout: {current_date} info of {district} "
                        f"(attempt {attempt + 1}/{max_retries})"
                    )

                    if attempt < max_retries - 1:
                        time.sleep(2)
            else:
                raise RuntimeError(
                    f"Failed after {max_retries} attempts: "
                    f"{current_date} info of {district}"
                )
        with engine.begin() as conn:
            print(f"Inserting {current_date} into DB")
            result = pd.concat(data, ignore_index=True).to_dict(orient="records")
            insert_individual_apt_sales(conn,result)

        current_date += pd.DateOffset(months=1)


individual_sales_apartment_main()
    


fetched 중구 info
fetched 종로구 info
fetched 용산구 info
fetched 성동구 info
fetched 광진구 info
fetched 동대문구 info
fetched 중랑구 info
fetched 성북구 info
fetched 강북구 info
fetched 도봉구 info
fetched 노원구 info
attempt 1: Total_rows exceeds 200: attempting again...numOfRows has been set to 385
fetched 은평구 info
fetched 서대문구 info
fetched 마포구 info
fetched 양천구 info
fetched 강서구 info
fetched 구로구 info
fetched 금천구 info
fetched 영등포구 info
fetched 동작구 info
fetched 관악구 info
fetched 서초구 info
fetched 강남구 info
fetched 송파구 info
fetched 강동구 info
Inserting 2023-03-01 00:00:00 into DB
fetched 중구 info
fetched 종로구 info
fetched 용산구 info
fetched 성동구 info
fetched 광진구 info
fetched 동대문구 info
fetched 중랑구 info
fetched 성북구 info
fetched 강북구 info
fetched 도봉구 info
fetched 노원구 info
fetched 은평구 info
fetched 서대문구 info
fetched 마포구 info
fetched 양천구 info
fetched 강서구 info
fetched 구로구 info
fetched 금천구 info
fetched 영등포구 info
fetched 동작구 info
fetched 관악구 info
fetched 서초구 info
fetched 강남구 info
fetched 송파구 info
fetched 강동구 info
Inserting 2023-04-01 00: